In [3]:
!pip install faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 96.7 MB/s eta 0:00:00


# Full pipeline

In [8]:
import os
import json
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util
import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import re

nltk.download('punkt')
nltk.download('punkt_tab')

# load topics

def extract_topics(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    topics = []
    for category, item_lists in data.items():
        for item_list in item_lists:
            topics.append({
                'category': category,
                'topic': item_list[0],
                'year': item_list[1]
            })
    return topics

topics = extract_topics('topic_results_with_year.json') # mettere path corretto

def load_json_documents(folder_path):
    documents = {}  # key = doc_name, value = text
    for file in os.listdir(folder_path):
        if file.endswith('.json'):
            with open(os.path.join(folder_path, file), 'r', encoding='utf-8') as f:
                data = json.load(f)

                if isinstance(data, dict):
                    for key, text in data.items():
                        documents[key] = text

                elif isinstance(data, list):
                    # assume each element is a dict with one key-value
                    for idx, item in enumerate(data):
                        if isinstance(item, dict):
                            for key, text in item.items():
                                # make a unique key if needed
                                documents[f"{file}_{key}_{idx}"] = text
                        else:
                            # fallback: store as string
                            documents[f"{file}_{idx}"] = str(item)

                else:
                    # fallback: store as string
                    documents[file] = str(data)

    return documents

documents = load_json_documents('/content/drive/MyDrive/PhD/safety_and_rag/json final') # mettere path corretto

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Loaded 7629 topics from 5 categories
Loaded 26985 documents


In [33]:
#split documents into sentences
doc_sentences = {}
doc_metadata = []

for doc_name, text in documents.items():
    sentences = sent_tokenize(text)
    doc_sentences[doc_name] = sentences
    for s in sentences:
        doc_metadata.append({'document': doc_name, 'sentence': s})

all_sentences = [m['sentence'] for m in doc_metadata]


# define cleaning filters

def clean_text(text):
    # remove bullet points and repeated whitespace
    text = re.sub(r'[•○*]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    # remove navigation or common headers (if any known patterns)
    text = re.sub(r'Go to main content', '', text, flags=re.IGNORECASE)
    return text.strip()

# apply filters to all sentences
for doc_name, sents in doc_sentences.items():
    doc_sentences[doc_name] = [clean_text(s) for s in sents]

doc_metadata = []
for doc_name, sents in doc_sentences.items():
    for s in sents:
        doc_metadata.append({'document': doc_name, 'sentence': s})

all_sentences = [m['sentence'] for m in doc_metadata]


def is_valid_sentence(s):
    # remove extra space
    s = s.strip()
    # skip if too short
    if len(s) < 30:
        return False
    # skip if mostly non-alphabetic
    alpha_ratio = sum(c.isalpha() for c in s) / max(len(s), 1)
    if alpha_ratio < 0.5:
        return False
    # skip common header pattern
    if re.search(r'English.*العربية.*Español.*Français', s):
        return False
    return True

# apply filter
for doc_name, sents in doc_sentences.items():
    doc_sentences[doc_name] = [clean_text(s) for s in sents if is_valid_sentence(clean_text(s))]

doc_metadata = []
for doc_name, sents in doc_sentences.items():
    for s in sents:
        doc_metadata.append({'document': doc_name, 'sentence': s})

all_sentences = [m['sentence'] for m in doc_metadata]


# compute embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

# topic embeddings
topic_texts = [t['topic'] for t in topics]
topic_embeddings = model.encode(topic_texts, convert_to_numpy=True, show_progress_bar=True)

# docs sentence embeddings
sentence_embeddings = model.encode(all_sentences, convert_to_numpy=True, show_progress_bar=True)

# Normalize for cosine similarity
faiss.normalize_L2(topic_embeddings)
faiss.normalize_L2(sentence_embeddings)

# faiss index to search more quickly
embedding_dim = sentence_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(sentence_embeddings)

# actual semantic search
top_k = 3  # top 3 matches per topic
threshold = 0.55  # similarity threshold, possiamo cambiarlo

results = []

for i, topic in enumerate(tqdm(topic_texts, desc='Matching topics')):
    topic_vec = np.expand_dims(topic_embeddings[i], axis=0).astype('float32')
    scores, indices = index.search(topic_vec, top_k)

    for j in range(top_k):
        score = float(scores[0][j])
        idx = int(indices[0][j])
        if score >= threshold:
            results.append({
                'topic': topic_texts[i],
                'category': topics[i]['category'],
                'year': topics[i]['year'],
                'document': doc_metadata[idx]['document'],
                'sentence': doc_metadata[idx]['sentence'],
                'score': score
            })

results_df = pd.DataFrame(results)
print(f'Total matches above threshold: {len(results_df)}')

# do aggregation
doc_summary = results_df.groupby('document').agg(
    num_topics_covered=('topic', 'nunique'),
    avg_score=('score', 'mean')
).reset_index()

doc_summary['coverage_pct'] = doc_summary['num_topics_covered'] / len(topics) * 100

doc_summary = doc_summary.sort_values(by='coverage_pct', ascending=False)
print(doc_summary.head())


doc_summary.to_csv('document_topic_coverage_summary.csv', index=False)
results_df.to_csv('document_topic_matches.csv', index=False)

KeyboardInterrupt: 